# GrapPA Standalone

In order to pretrain the GrapPA components of the reconstruction chain, we need to use DBSCAN-based or label-based cluster labels for GrapPA-shower, GrapPA-track and GrapPA inter. This notebook is used to check the the labels look sensible:
- The start points of shower fragments make intuitive sense
- The end points of track fragments make intuitive sense
- The end points of the particle labels make intuitive sense
- The primary ID of particles make intuitive sense
- The particle ID of particles make intuitive sense

## GrapPA-Shower Labels

In [20]:
import sys
# sys.path.append('/sdf/group/neutrino/drielsma/dev/spine')
# sys.path.append('/app/data/spine_bilal/spine/')

import sys

SOFTWARE_DIR = '/app/data/spine/' # Change this path to your software install

# Set software directory
sys.path.append(SOFTWARE_DIR)

import yaml

from spine.driver import Driver
from spine.utils.inference import get_inference_cfg

# Define the data loader configuration
cfg = get_inference_cfg(cfg='/app/data/ml_reco_2025_04_09/standalone_francois/grappa_shower_rep2_zd_e_1/3_grappa_shower_rep2_zd_e_1.cfg',
                        file_keys='/app/data/inference_test/test.root',
                        weight_path='/app/data/ml_reco_2025_04_09/standalone_francois/grappa_shower_rep2_zd_e_1/*.ckpt',
                        batch_size=1, num_workers=0)

# Edit config on the fly to change node labeling
cfg['base']['world_size'] = 0
cfg['model']['loss_input']['coord_label'] = 'coord_label'
cfg['model']['modules']['grappa_loss']['node_loss']['high_purity'] = True
# cfg['model']['modules']['grappa_loss']['node_loss']['use_closest'] = True

# Add meta loading
cfg['io']['loader']['dataset']['schema']['meta'] = {
        'parser': 'meta',
        'sparse_event':'sparse3d_pcluster'
}

# Prepare driver
driver = Driver(cfg)
print('Number of events in set:', len(driver))


 ██████████   ██████████    ███   ███       ██   ███████████
███        █  ██       ███   █    █████     ██   ██         
  ████████    ██       ███  ███   ██  ████  ██   ██████████ 
█        ███  ██████████     █    ██     █████   ██         
 ██████████   ██            ███   ██       ███   ███████████

Release version: 0.2.2

$CUDA_VISIBLE_DEVICES=

Configuration processed at: Linux c60caec20b06 5.15.167.4-microsoft-standard-WSL2 #1 SMP Tue Nov 5 00:21:55 UTC 2024 x86_64 x86_64 x86_64 GNU/Linux

base: {epochs: 21, log_dir: /lus/eagle/clone/g2/projects/Nu_Novel/zel_train_weights/logs/grappa_shower_rep2_e_1,
  log_step: 1, overwrite_log: true, seed: 0, unwrap: true, world_size: 0}
io:
  loader:
    batch_size: 1
    collate_fn: all
    dataset:
      file_keys: /app/data/inference_test/test.root
      name: larcv
      schema:
        coord_label: {cluster_event: cluster3d_pcluster, parser: particle_coords,
          particle_event: particle_pcluster}
        data: {add_particle_info:

In [21]:
import numpy as np

# entries = [1]
# driver.apply_filter(entry_list=entries)
entry = 0

data = driver.process()

print(f"Keys: {data.keys()}")

accuracies = []
losses = []
node_losses = []
node_accuracies = []
edge_losses = []
edge_accuracies = []

for i in range(100):
  data = driver.process()

  accuracies.append(data['accuracy'])
  losses.append(data['loss'])
  node_losses.append(data['node_loss'])
  node_accuracies.append(data['node_accuracy'])
  edge_losses.append(data['edge_loss'])
  edge_accuracies.append(data['edge_accuracy'])

Keys: dict_keys(['index', 'file_index', 'file_entry_index', 'coord_label', 'data', 'meta', 'clusts', 'edge_index', 'start_points', 'end_points', 'node_pred', 'edge_pred', 'group_pred', 'node_accuracy', 'node_loss', 'node_count', 'edge_accuracy', 'edge_loss', 'edge_count', 'loss', 'accuracy'])


In [22]:
from spine.utils.globals import GROUP_COL
from spine.utils.metrics import eff, pur, pur_eff, ari
def francois_metrix(data):
  
  voxel_label = data['data'][0][:, GROUP_COL]
  voxel_pred = np.full_like(voxel_label, -1)
  for i, c in enumerate(data['clusts'][0]):
    voxel_pred[c] = data['group_pred'][0][i]

  mask = voxel_pred > -1
  return (eff(voxel_label[mask], voxel_pred[mask]), ari(voxel_label[mask], voxel_pred[mask]), pur(voxel_label[mask], voxel_pred[mask]))

def bilal_metrix(data):
  # Extract cluster labels correctly
  cluster_label = data['data'][entry]

  # Extract voxel labels correctly
  voxel_label = cluster_label[:, GROUP_COL]  # Extract GROUP_COL from the NumPy array

  # **Fix: Flatten `voxel_label` if it has extra dimensions**
  if voxel_label.ndim > 1:
      voxel_label = voxel_label.flatten()

  # **Fix: Ensure `voxel_pred` is correctly sized**
  max_index = max(max(c) for c in data['clusts'][entry]) if data['clusts'][entry] else 0
  voxel_pred = np.full((max_index + 1,), -1)  # Ensure correct size

  # Assign cluster predictions
  for i, c in enumerate(data['clusts'][entry]):
      voxel_pred[c] = data['group_pred'][entry][i]

  # **Fix: Trim `voxel_label` or `voxel_pred` to the same size**
  min_length = min(len(voxel_label), len(voxel_pred))
  voxel_label = voxel_label[:min_length]
  voxel_pred = voxel_pred[:min_length]

  # **Ensure mask does not cause mismatches**
  mask = (voxel_pred > -1) & (voxel_label > -1)

  # Compute Adjusted Rand Index (ARI)
  ari_score = ari(voxel_label[mask], voxel_pred[mask])
  # print(f"Adjusted Rand Index (ARI): {ari_score}")

  # Compute assignment efficiency
  efficiency = eff(voxel_label[mask], voxel_pred[mask])
  # print('Assignment Efficiency:', efficiency)

  # Compute purity and efficiency
  purity, efficiency = pur_eff(voxel_label[mask], voxel_pred[mask])
  # print('Assignment Purity:', purity)

  return (efficiency, ari_score, purity)

In [23]:
from spine.utils.gnn.cluster import get_cluster_label, get_cluster_closest_primary_label
from spine.utils.globals import GROUP_COL
from spine.utils.metrics import eff, pur_eff, ari

num_samples = 1000  
bilal_metrixs = []
francois_metrixs = []

for i in range(num_samples):
  data = driver.process()

  bilal_metrixs.append(bilal_metrix(data))
  francois_metrixs.append(francois_metrix(data))

bilal_metrixs = np.array(bilal_metrixs)
francois_metrixs = np.array(francois_metrixs)

bilal_eff_avg = np.average(bilal_metrixs[:, 0])
bilal_ari_avg = np.average(bilal_metrixs[:, 1])
bilal_pur_avg = np.average(bilal_metrixs[:, 2])
francois_eff_avg = np.average(francois_metrixs[:, 0])
francois_ari_avg = np.average(francois_metrixs[:, 1])
francois_pur_avg = np.average(francois_metrixs[:, 2])

print("Bilal Metrix: ")
print(f"Effeciency: {bilal_eff_avg} ")
print(f"ARI: {bilal_ari_avg} ")
print(f"Purity: {bilal_pur_avg} ")
print("\nFrancois Metrix: ")
print(f"Effeciency: {francois_eff_avg} ")
print(f"ARI: {francois_ari_avg} ")
print(f"Purity: {francois_pur_avg} ")

Bilal Metrix: 
Effeciency: 0.7942793291178034 
ARI: 0.581029790078748 
Purity: 0.966 

Francois Metrix: 
Effeciency: 0.7942793291178034 
ARI: 0.581029790078748 
Purity: 0.966 


In [ ]:
import numpy as np
print('Averages')

print(f'Accuracy: {np.average(accuracies)}')
print(f'Loss: {np.average(losses)}')
print(f'Node Loss: {np.average(node_losses)}')
print(f'Node Accuracy: {np.average(node_accuracies)}')
print(f'Edge Loss: {np.average(edge_losses)}')
print(f'Edge Accuracy: {np.average(edge_accuracies)}')

In [ ]:
# Let's get the training data

import pandas as pd
import os
import matplotlib.pyplot as plt

log_path = '/home/jamars/MLReco/lartpc_mlreco3d/logs/cnn_encoder/edge/nominal/grappa_shower'
log_files = ['train_log-0000000.csv', 'train_log-0004000.csv', 'train_log-0008000.csv', 'train_proc0_log-0012000.csv', 'train_proc0_log-0015000.csv', 'train_proc0_log-0018000.csv', 'train_proc0_log-0021000.csv']

iters = []
acc = []
loss = []



for f in log_files:
  d = pd.read_csv(os.path.join(log_path, f))

  iters.append(d['iter'])
  acc.append(d['accuracy'])
  loss.append(d['loss'])

iters = np.concatenate(iters)
acc = np.concatenate(acc)
loss = np.concatenate(loss)

plt.plot(iters, acc)
plt.plot(iters, loss)



In [ ]:
latest_data = pd.read_csv(os.path.join(log_path, log_files[-1]))

print(f'Average: {np.average(latest_data["accuracy"])}')
print(f'Loss: {np.average(latest_data["loss"])}')
print(f'Node Loss: {np.average(latest_data["node_loss"])}')
print(f'Node Accuracy: {np.average(latest_data["node_accuracy"])}')
print(f'Edge Loss: {np.average(latest_data["edge_loss"])}')
print(f'Edge Accuracy: {np.average(latest_data["edge_accuracy"])}')

In [ ]:
# Printing the metrics that we are concerned with (purity, ARI, effeciency)
from spine.utils.metrics import pur, ari, eff
from scipy.special import softmax
from spine.vis import scatter_points, scatter_clusters, network_topology
from spine.vis.layout import HIGH_CONTRAST_COLORS
from spine.utils.gnn.cluster import get_cluster_label, get_cluster_closest_primary_label
from spine.utils.gnn.evaluation import node_assignment_score, node_purity_mask
from spine.utils.globals import COORD_COLS, GROUP_COL

# Get shower labels
cluster_label = data['data'][entry]
voxels = cluster_label[:, COORD_COLS]
clusts = data['clusts'][entry]
edge_index = data['edge_index'][entry]
edge_pred = data['edge_pred'][entry]
node_pred = data['node_pred'][entry]
shower_preds = data['group_pred'][entry]
group_label = get_cluster_label(data['data'][entry], clusts, column=GROUP_COL)

voxel_label = data['data'][entry][:, GROUP_COL]
voxel_pred = np.full_like(voxel_label, -1)
for i, c in enumerate(clusts):
    voxel_pred[c] = shower_preds[i]

mask = voxel_pred > -1
a = ari(voxel_label[mask], voxel_pred[mask])

In [ ]:
from spine.vis import layout3d
from plotly import graph_objs as go
from plotly.offline import iplot

# camera = dict(eye=dict(x=1.5,y=0.75,z=1.5), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.25, z=0.1))
camera = dict(eye=dict(x=1.75,y=0.875,z=0.02), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.125, z=0.02)) # Default

# layout = layout3d(detector='2x2', camera=camera)
layout = layout3d(detector='2x2', meta=data['meta'][entry], camera=camera)

In [ ]:
from spine.vis import GeoDrawer

tpc_traces = GeoDrawer('2x2', detector_coords=False).tpc_traces(meta=data['meta'][entry])

### Input node features

In [ ]:
from spine.vis import scatter_points, scatter_clusters 
from spine.vis.layout import HIGH_CONTRAST_COLORS
from spine.utils.globals import COORD_COLS

# Get shower labels
cluster_label = data['data'][entry]
voxels = cluster_label[:, COORD_COLS]
start_points = data['start_points'][entry]
end_points = data['end_points'][entry]
clusts = data['clusts'][entry]

# Draw fragments
graph = []
graph += scatter_clusters(voxels, clusts, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Fragments')

# Draw input end points
graph += scatter_points(start_points, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle')
graph[-1]['name'] = 'Start points'
graph += scatter_points(end_points, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle-open')
graph[-1]['name'] = 'End points'

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)

### Predictions

In [ ]:
from scipy.special import softmax
from spine.vis import scatter_points, scatter_clusters, network_topology
from spine.vis.layout import HIGH_CONTRAST_COLORS
from spine.utils.gnn.cluster import get_cluster_label, get_cluster_closest_primary_label
from spine.utils.gnn.evaluation import node_assignment_score, node_purity_mask
from spine.utils.metrics import pur, ari, eff
from spine.utils.globals import COORD_COLS, SHAPE_COL, PART_COL, GROUP_COL, PRGRP_COL

# Get shower labels
cluster_label = data['data'][entry]
voxels = cluster_label[:, COORD_COLS]
clusts = data['clusts'][entry]
edge_index = data['edge_index'][entry]
edge_pred = data['edge_pred'][entry]
node_pred = data['node_pred'][entry]
shower_preds = data['group_pred'][entry]
group_label = get_cluster_label(data['data'][entry], clusts, column=GROUP_COL)

voxel_label = data['data'][entry][:, GROUP_COL]
voxel_pred = np.full_like(voxel_label, -1)
for i, c in enumerate(clusts):
    voxel_pred[c] = shower_preds[i]

mask = voxel_pred > -1
a = ari(voxel_label[mask], voxel_pred[mask])

print(a)

# Draw fragments
graph = []
graph += scatter_clusters(voxels, clusts, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Fragments')

shower_parts = get_cluster_label(cluster_label, clusts, PART_COL)
graph += scatter_clusters(voxels, clusts, color=shower_parts, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Particles')

# Draw shower truth info
shower_shapes = get_cluster_label(cluster_label, clusts, SHAPE_COL)
shower_labels = get_cluster_label(cluster_label, clusts, GROUP_COL)

shower_primary_labels = get_cluster_label(cluster_label, clusts, PRGRP_COL)
shower_primary_labels = get_cluster_closest_primary_label(cluster_label, data['coord_label'][entry], clusts, shower_primary_labels) # Comment out for old labeling scheme
hp_mask = node_purity_mask(shower_preds, shower_primary_labels)
shower_primary_labels_masked = shower_primary_labels.copy()
shower_primary_labels_masked[~hp_mask] = -1

# shower_primary_labels_masked = data['node_label'][entry]
# print((shower_primary_labels == data['node_label'][entry]).all())
graph += scatter_clusters(voxels, clusts, color=shower_shapes, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Semantics')
graph += scatter_clusters(voxels, clusts, color=shower_labels, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Labels')
graph += scatter_clusters(voxels, clusts, color=shower_primary_labels, colorscale='Portland', cmin=-1, cmax=1, markersize=2, name='Shower Primary Labels')
graph += scatter_clusters(voxels, clusts, color=shower_primary_labels_masked, colorscale='Portland', cmin=-1, cmax=1, markersize=2, name='Shower Primary Labels (masked)')

# Draw shower predictions
# shower_preds = np.unique(node_assignment_score(edge_index, edge_pred, len(clusts)), return_inverse=True)[-1]
shower_primary_preds = softmax(node_pred, axis=1)[:,-1]
shower_edge_mask = np.argmax(edge_pred, axis=-1).astype(bool)
shower_edge_index = edge_index[shower_edge_mask]
graph += network_topology(voxels, clusts, shower_edge_index, clust_labels=shower_preds, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Shower Predictions')
graph += scatter_clusters(voxels, clusts, color=shower_primary_preds, colorscale='Portland', cmin=0, cmax=1, markersize=2, name='Shower Primary Predictions')

graph += tpc_traces

print('Edge accuracy', data['edge_accuracy'])
print('Node accuracy', data['node_accuracy'])
fig = go.Figure(graph, layout=layout)
iplot(fig)

## GrapPA-Track Labels

In [ ]:
import sys
sys.path.append('/sdf/group/neutrino/drielsma/dev/spine')

import yaml

from spine.driver import Driver
from spine.utils.inference import get_inference_cfg

# Define the data loader configuration
cfg = get_inference_cfg(cfg='/sdf/data/neutrino/2x2/spine/train/mpvmpr_v2/config/grappa_track/grappa_track.cfg',
                        file_keys='/sdf/data/neutrino/2x2/sim/mpvmpr_v2/test_file_list.txt',
                        weight_path='/sdf/data/neutrino/2x2/spine/train/mpvmpr_v2/weights/grappa_track/default/snapshot-49999.ckpt',
                        batch_size=1, num_workers=0)

# Add meta loading
cfg['io']['loader']['dataset']['schema']['meta'] = {
        'parser': 'meta',
        'sparse_event':'sparse3d_pcluster'
}

# Prepare driver
driver = Driver(cfg)
print('Number of events in set:', len(driver))

In [ ]:
import numpy as np

entries = [0]
driver.apply_filter(entry_list=entries)
entry = 0

data = driver.process()

In [ ]:
from spine.vis import layout3d
from plotly import graph_objs as go
from plotly.offline import iplot

# camera = dict(eye=dict(x=1.5,y=0.75,z=1.5), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.25, z=0.1))
camera = dict(eye=dict(x=1.75,y=0.875,z=0.02), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.125, z=0.02)) # Default

layout = layout3d(detector='2x2', meta=data['meta'][entry], camera=camera)

In [ ]:
from spine.vis import GeoDrawer

tpc_traces = GeoDrawer('2x2', detector_coords=False).tpc_traces(meta=data['meta'][entry])

### Input node features

In [ ]:
from spine.vis import scatter_points, scatter_clusters 
from spine.vis.layout import HIGH_CONTRAST_COLORS
from spine.utils.globals import COORD_COLS

# Get track labels
cluster_label = data['data'][entry]
voxels = cluster_label[:, COORD_COLS]
start_points = data['start_points'][entry]
end_points = data['end_points'][entry]
clusts = data['clusts'][entry]

# Draw fragments
graph = []
graph += scatter_clusters(voxels, clusts, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Track Fragments')

# Draw input end points
graph += scatter_points(start_points, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle')
graph[-1]['name'] = 'Start points'
graph += scatter_points(end_points, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle-open')
graph[-1]['name'] = 'End points'

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)

### Predictions

In [ ]:
from scipy.special import softmax
from spine.vis import scatter_points, scatter_clusters, network_topology
from spine.vis.layout import HIGH_CONTRAST_COLORS
from spine.utils.gnn.cluster import get_cluster_label
from spine.utils.gnn.evaluation import node_assignment_score
from spine.utils.globals import COORD_COLS, SHAPE_COL, GROUP_COL, PRGRP_COL

# Get track labels
cluster_label = data['data'][entry]
voxels = cluster_label[:, COORD_COLS]
clusts = data['clusts'][entry]
edge_index = data['edge_index'][entry]
edge_pred = data['edge_pred'][entry]
node_pred = data['node_pred'][entry]

# Draw fragments
graph = []
graph += scatter_clusters(voxels, clusts, color=np.arange(len(clusts)), colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Track Fragments')

# Draw track truth info
track_labels = get_cluster_label(cluster_label, clusts, GROUP_COL)
graph += scatter_clusters(voxels, clusts, color=track_labels, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Track Labels')

# Draw track predictions
track_preds = np.unique(node_assignment_score(edge_index, edge_pred, len(clusts)), return_inverse=True)[-1]
track_edge_mask = np.argmax(edge_pred, axis=-1).astype(bool)
track_edge_index = edge_index[track_edge_mask]
graph += network_topology(voxels, clusts, track_edge_index, clust_labels=track_preds, colorscale=HIGH_CONTRAST_COLORS, markersize=2, name='Track Predictions')

graph += tpc_traces

print('Edge accuracy', data['edge_accuracy'])
fig = go.Figure(graph, layout=layout)
iplot(fig)

## GrapPA-Inter

In [ ]:
import sys
import yaml
sys.path.append('/sdf/group/neutrino/drielsma/dev/spine')
from spine.driver import Driver
from spine.utils.inference import get_inference_cfg

# Define the data loader configuration
cfg = get_inference_cfg(cfg='/sdf/data/neutrino/2x2/spine/train/mpvmpr_v2/config/grappa_inter/grappa_inter.cfg',
                        file_keys='/sdf/data/neutrino/2x2/sim/mpvmpr_v2/test_file_list.txt',
                        weight_path='/sdf/data/neutrino/2x2/spine/train/mpvmpr_v2/weights/grappa_inter/default/snapshot-24999.ckpt',
                        batch_size=1, num_workers=0)

# Parse true particle information
cfg['io']['loader']['dataset']['schema']['particles'] = {
        'parser': 'particle',
        'particle_event': 'particle_pcluster',
        'cluster_event':'cluster3d_pcluster'
}

# Parse metatadata information
cfg['io']['loader']['dataset']['schema']['meta'] = {
        'parser': 'meta',
        'sparse_event':'sparse3d_pcluster'
}

# Instantiate driver
driver = Driver(cfg)

In [ ]:
import numpy as np

entry_list = [0]
driver.apply_filter(entry_list=entry_list)

data = driver.process()
entry = 0

print(data['node_type_accuracy'])

In [ ]:
data.keys()

In [ ]:
from spine.utils.globals import COORD_COLS, VALUE_COL, SHAPE_COL

voxels = data['data'][entry][:, COORD_COLS]
charges = data['data'][entry][:, VALUE_COL]

In [ ]:
import numpy as np
from plotly import graph_objs as go
from plotly.offline import iplot

from spine.vis import layout3d

# camera = dict(eye=dict(x=1.75,y=0.875,z=0.02), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.125, z=0.02)) # Default
camera = dict(eye=dict(x=1.75,y=0.875,z=0.02), up=dict(x=0,y=1,z=0), center=dict(x=0, y=-0.125, z=0.02)) # Default

layout = layout3d(detector='2x2', meta=data['meta'][entry], camera=camera)
markersize = 2

In [ ]:
from spine.vis import GeoDrawer

tpc_traces = GeoDrawer('2x2', detector_coords=False).tpc_traces(meta=data['meta'][entry])

### Truth information

In [ ]:
from spine.vis import scatter_particles

graph = scatter_particles(data['data'][entry], data['particles'][entry], markersize=markersize)

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)

### Start or end points of input particles

In [ ]:
from spine.vis import scatter_points, scatter_clusters
from spine.vis.layout import HIGH_CONTRAST_COLORS

# Get particles
particles = data['clusts'][entry]
start_points = data['start_points'][entry]
end_points = data['end_points'][entry]

# Draw particles
graph = []
graph += scatter_clusters(voxels, particles, colorscale=HIGH_CONTRAST_COLORS, markersize=markersize, name='Particles')

# Draw input end points
graph += scatter_points(start_points, color=np.arange(len(particles)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle', name='Start points')
graph += scatter_points(end_points, color=np.arange(len(particles)), colorscale=HIGH_CONTRAST_COLORS, markersize=7, marker_symbol='circle-open', name='End points')

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)

### Particle classification

In [ ]:
from scipy.special import softmax

from spine.utils.globals import CLUST_COL, GROUP_COL, INTER_COL, NU_COL, PID_COL, PRGRP_COL, PRINT_COL, VTX_COLS
from spine.utils.gnn.cluster import get_cluster_label
from spine.vis import scatter_points, scatter_clusters, network_topology
from spine.vis.layout import PLOTLY_COLORS, HIGH_CONTRAST_COLORS, PLOTLY_COLORS_WGRAY

# Get labels
cluster_label = data['data'][entry]
particles = data['clusts'][entry]

pid_labels = get_cluster_label(cluster_label, particles, column=PID_COL)
primary_labels = get_cluster_label(cluster_label, particles, column=PRINT_COL)

pid_preds = np.argmax(data['node_type_pred'][entry], axis=1)
primary_preds = softmax(data['node_primary_pred'][entry], axis=1)[:,-1]

# Draw particles
graph = []
graph += scatter_clusters(voxels, particles, colorscale=HIGH_CONTRAST_COLORS, markersize=markersize, name='Particles')

# Draw shower truth info
graph += scatter_clusters(voxels, particles, color=pid_labels, colorscale=PLOTLY_COLORS_WGRAY[:6], cmin=-1, cmax=4, markersize=markersize, name='PID Labels')
graph += scatter_clusters(voxels, particles, color=primary_labels, colorscale=PLOTLY_COLORS_WGRAY[:3], cmin=-1, cmax=1, markersize=markersize, name='Primary Labels')

# Draw shower predictions
graph += scatter_clusters(voxels, particles, color=pid_preds, colorscale=PLOTLY_COLORS_WGRAY[:6], cmin=-1, cmax=4, markersize=markersize, name='PID Predictions')
graph += scatter_clusters(voxels, particles, color=primary_preds, colorscale='Portland', cmin=0, cmax=1, markersize=markersize, name='Primary Predictions')

# vertex_preds = output['node_pred_vtx'][0][primary_mask,:3] * 6144
# graph += scatter_points(vertex_preds, color=np.arange(len(output['clusts'][0]))[primary_mask], colorscale=high_contrast_colorscale(), cmin=0, cmax=len(output['clusts'][0]), markersize=7)
# graph[-1]['name'] = 'Vertex predictions'

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)

In [ ]:
from spine.utils.globals import PID_LABELS

PID_LABELS

### Interaction clustering

In [ ]:
from scipy.special import softmax

from spine.utils.globals import CLUST_COL, GROUP_COL, INTER_COL, NU_COL, PID_COL, PRGRP_COL, PRINT_COL, VTX_COLS
from spine.utils.gnn.cluster import get_cluster_label
from spine.vis import scatter_points, scatter_clusters, network_topology
from spine.vis.layout import PLOTLY_COLORS, HIGH_CONTRAST_COLORS, PLOTLY_COLORS_WGRAY

# Get track labels
cluster_label = data['data'][entry]
particles = data['clusts'][entry]

inter_labels = get_cluster_label(cluster_label, particles, column=INTER_COL)

inter_preds = data['group_pred'][entry]

# Draw particles
graph = []
graph += scatter_clusters(voxels, particles, colorscale=HIGH_CONTRAST_COLORS, markersize=markersize, name='Particles')

# Draw interactioon truth info
graph += scatter_clusters(voxels, particles, color=inter_labels, colorscale=HIGH_CONTRAST_COLORS, markersize=markersize, name='Interaction Labels')

# Draw interaction predictions
graph += scatter_clusters(voxels, particles, color=inter_preds, colorscale=HIGH_CONTRAST_COLORS, markersize=markersize, name='Interaction Predictions')

graph += tpc_traces

fig = go.Figure(graph, layout=layout)
iplot(fig)